In [1]:
%pip install langgraph langchain langchain-openAI tavily-python graphviz matplotlib python-dotenv ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

# Specify the full path to your .env file
load_dotenv(r"C:\Users\Tendai\OneDrive\Chat Agents\.env")
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

In [3]:
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

# --- Documents ---
documents = [
    {
        "id": "q1",
        "text": "Question 1: Open sore, blister, or ulcer on either foot? An open sore is a break in the skin that hasn't healed — it may look like a raw patch, a wound, or a crater. A blister is a bubble of fluid under the skin from friction or pressure. An ulcer is a deeper, slower-healing wound often found on the bottom of the foot in diabetic patients.",
        "metadata": {"question": "Question 1: Open sore, blister, or ulcer on either foot?"}
    },
    {
        "id": "q2",
        "text": "Question 2: Any new redness, warmth or swelling today? Redness means the skin looks pinker or redder than normal, especially compared to the other foot. Warmth means an area feels noticeably hotter to the touch than surrounding skin. Swelling means a part of the foot looks or feels puffy and larger than usual.",
        "metadata": {"question": "Question 2: Any new redness, warmth or swelling today?"}
    },
    {
        "id": "q3",
        "text": "Question 3: Any drainage, odor, or bleeding from the foot or toes? Drainage means any fluid leaking from the skin. Odor means an unusual or unpleasant smell coming from the foot, which can signal infection. Bleeding means any active or dried blood on the foot or toes.",
        "metadata": {"question": "Question 3: Any drainage, odor, or bleeding from the foot or toes?"}
    }
]

# --- Embed ---
def get_embedding(text: str) -> np.ndarray:
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return np.array(response.data[0].embedding)

# Embed all documents and store
doc_embeddings = np.array([get_embedding(doc["text"]) for doc in documents])

In [4]:
# --- Retrieve ---
def retrieve(query: str) -> dict:
    query_embedding = get_embedding(query)
    
    # Cosine similarity: dot product of normalized vectors
    query_norm = query_embedding / np.linalg.norm(query_embedding)
    doc_norms = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    scores = doc_norms @ query_norm
    
    best_idx = np.argmax(scores)
    return documents[best_idx]
# --- Example Usage ---
query = "I'm confused about question 1"
result = retrieve(query)
print(result["metadata"]["question"])
print(result["text"])

Question 2: Any new redness, warmth or swelling today?
Question 2: Any new redness, warmth or swelling today? Redness means the skin looks pinker or redder than normal, especially compared to the other foot. Warmth means an area feels noticeably hotter to the touch than surrounding skin. Swelling means a part of the foot looks or feels puffy and larger than usual.
